# 4.5 Type Hints for Functions

**Prerequisites:** 4.1 Functions User-defined, 4.4 Function Decorator  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What a type hint is, and what it is emphatically not
- Annotating parameters, defaults and return values
- Container types, and the 3.9+ builtin generics
- `X | None` for optional values, and why `Optional` means that too
- `Callable` — annotating functions that take functions
- `Any`, `TypeAlias`, and the 3.12 `type` statement
- Running `mypy`, and reading what it tells you
- How much typing is worth it

---

## Why annotate a function at all?

Python is **dynamically typed** — it checks types at run time, when an operation actually
happens (see **1.1**). That is not going to change. Type hints do not turn Python into Java.

What they do is let you *write down* what you already know:

```python
def apply_discount(price, percent):     # price in rupees? paise? a Decimal? a list?
def apply_discount(price: float, percent: float = 10.0) -> float:
```

The second version answers three questions the first leaves open, and — unlike a comment —
a tool can check that the answer is still true.

### Three concrete payoffs

| Benefit | Why it matters |
|---|---|
| **Documentation that can't rot** | A comment saying "price is a float" can drift out of date silently. An annotation is checked. |
| **Editor support** | Autocomplete, "go to definition" and inline errors all get dramatically better. |
| **Bugs caught before running** | `mypy` finds the `None` you forgot to handle, without executing anything. |

> **Version note:** annotation *syntax* landed in **3.0**; PEP 484 gave it meaning in **3.5**;
> builtin generics (`list[int]` rather than `typing.List[int]`) in **3.9**; `X | Y` unions in
> **3.10**; and the `type` statement plus new generic syntax in **3.12**.

This notebook covers annotating **functions**. The full static-typing story — protocols,
generics, `TypedDict`, strictness settings — is **17 Type Hints and Static Typing**.

---

## 1. ⚠️ The most important fact: nothing is enforced

Read this before anything else, because it is the single most common misunderstanding.

**Python ignores type hints at run time.** They are stored on the function and otherwise
have no effect. Annotating a parameter `int` does not stop anyone passing a string, and does
not raise anything when they do.

The error, if there is one, surfaces later — wherever the wrong type finally breaks an
operation. Or never, if the operation happens to work.

In [ ]:
def add(a: int, b: int) -> int:
    return a + b

# The annotations are honoured by nobody
print("ints  :", add(2, 3))
print("floats:", add(2.5, 3.5), "   <- not ints, no complaint")
print("strs  :", add("py", "thon"), " <- + works on str, so this 'succeeds'")
print("lists :", add([1], [2]), "     <- and on lists")

# It only breaks where the OPERATION breaks, not where the type is wrong
try:
    add("2", 3)
except TypeError as exc:
    print("\nstr + int:", exc, " <- the annotation didn't catch this; `+` did")

# Where the annotations actually live
print("\n__annotations__:", add.__annotations__)

# You can read them at run time if you want to
for name, hint in add.__annotations__.items():
    label = "returns" if name == "return" else f"param {name}"
    print(f"  {label:<10} -> {hint.__name__}")

# ⚠️ Annotations are not even validated as types
def nonsense(x: "a hint can be any expression at all") -> 42:
    return x

print("\nnonsense still runs:", nonsense(1), "| hints:", nonsense.__annotations__)

---

## 2. Annotating parameters and returns

### Syntax breakdown

```
def greet(name: str, times: int = 1) -> str:
             |    |         |    |      |
             |    |         |    |      +-- return annotation
             |    |         |    +--------- default value (after the annotation)
             |    |         +-------------- parameter annotation
             |    +------------------------ parameter annotation
             +----------------------------- parameter name
```

Note the order when a parameter has both: **`name: type = default`**.

| Return case | Annotation |
|---|---|
| Returns a value | that value's type — `-> str` |
| Returns nothing useful | `-> None` |
| Never returns (always raises, or loops forever) | `-> NoReturn` |

In [ ]:
from typing import NoReturn

def greet(name: str, times: int = 1) -> str:
    """Return a greeting repeated `times` times."""
    return " ".join([f"Hello, {name}!"] * times)

def log(message: str) -> None:
    """Print a message. Returns nothing."""
    print(f"  LOG: {message}")

def fail(reason: str) -> NoReturn:
    """Always raises - never returns to the caller."""
    raise RuntimeError(reason)

print(greet("Aditya"))
print(greet("Priya", times=2))
log("annotated")

try:
    fail("deliberate")
except RuntimeError as exc:
    print("  fail() raised:", exc)


# Annotating *args and **kwargs: annotate ONE value, not the container
def summarise(*values: int, **options: str) -> str:
    return f"{len(values)} values, options={options}"

print("\n" + summarise(1, 2, 3, sep=",", style="short"))
print("annotations:", summarise.__annotations__)
print("  ^ *values: int means 'each value is an int', so values is a tuple[int, ...]")


# Positional-only and keyword-only annotate exactly the same way
def distance(x: float, y: float, /, *, unit: str = "km") -> str:
    return f"{(x**2 + y**2) ** 0.5:.2f} {unit}"

print("\n" + distance(3, 4, unit="miles"))

---

## 3. Container types

A bare `list` says almost nothing — a list of *what*? Generic annotations say both.

```
list[int]              a list of integers
dict[str, float]       keys are str, values are float
tuple[int, str]        exactly two items, of those types
tuple[int, ...]        any number of ints
set[str]               a set of strings
```

> **Version note — use the builtins.** Before **3.9** you had to write `typing.List[int]`,
> `typing.Dict[str, int]` and so on. From 3.9 the builtin types are subscriptable directly,
> and the `typing` aliases are deprecated. **Write `list[int]`, not `List[int]`.**

For parameters, the modern advice is to annotate what you actually *need*:

| Annotation | Accepts | Use when |
|---|---|---|
| `list[int]` | Only a list | You will mutate it, or index it |
| `Sequence[int]` | list, tuple, str... | You only read and index |
| `Iterable[int]` | anything you can `for` over | You only iterate, once |

In [ ]:
from collections.abc import Iterable, Sequence, Mapping

def average(marks: list[float]) -> float:
    return sum(marks) / len(marks)

def tally(words: Iterable[str]) -> dict[str, int]:
    counts: dict[str, int] = {}          # variables can be annotated too
    for word in words:
        counts[word] = counts.get(word, 0) + 1
    return counts

def split_name(full: str) -> tuple[str, str]:
    first, _, last = full.partition(" ")
    return first, last

def coordinates() -> list[tuple[float, float]]:
    return [(0.0, 0.0), (3.0, 4.0)]


print("average :", average([88, 91, 79]))
print("tally   :", tally(["a", "b", "a"]))
print("split   :", split_name("Aditya Tripathi"))
print("coords  :", coordinates())


# Iterable is more permissive than list - all of these work
print("\ntally over a list :", tally(["x", "y", "x"]))
print("tally over a tuple:", tally(("x", "y")))
print("tally over a str  :", tally("aab"))
print("tally over a gen  :", tally(w for w in ["p", "q", "p"]))

# Sequence: indexable and sized, but not necessarily mutable
def middle(items: Sequence[int]) -> int:
    return items[len(items) // 2]

print("\nmiddle of list :", middle([1, 2, 3]))
print("middle of tuple:", middle((1, 2, 3)))


# tuple[int, ...] means "any number of ints"
def total(*nums: int) -> int:
    return sum(nums)

def stats(values: tuple[int, ...]) -> Mapping[str, int]:
    return {"min": min(values), "max": max(values), "sum": sum(values)}

print("\nstats:", stats((3, 1, 4, 1, 5)))

---

## 4. Optional values: `X | None`

The most valuable thing type hints do in practice is force you to be explicit about **`None`**.

A function that "returns a user, or `None` if not found" is a bug factory when undocumented —
callers forget the `None` case and get `AttributeError` in production. Annotating it makes
the omission visible to a checker.

```python
def find_user(uid: int) -> str | None:      # 3.10+
def find_user(uid: int) -> Optional[str]:   # older; means exactly the same
```

`Optional[str]` is **not** "this argument is optional" — it means "`str` **or** `None`".
An argument with a default is optional; that's a separate idea.

> **Version note:** `X | Y` union syntax requires **3.10+**. Below that, use
> `typing.Optional[X]` and `typing.Union[X, Y]`.

In [ ]:
from typing import Optional, Union

USERS = {1: "Aditya", 2: "Priya"}

# Modern (3.10+)
def find_user(uid: int) -> str | None:
    return USERS.get(uid)

# Older spelling - identical meaning
def find_user_old(uid: int) -> Optional[str]:
    return USERS.get(uid)

print("found  :", find_user(1))
print("missing:", find_user(99))
print("\nOptional[str] IS str | None:", Optional[str] == Union[str, None])


# The bug the annotation makes visible
def greet_unsafe(uid: int) -> str:
    name = find_user(uid)
    return f"Hello, {name.upper()}"          # mypy: "str | None" has no attribute "upper"

try:
    greet_unsafe(99)
except AttributeError as exc:
    print("\nat run time:", exc)

# The fix: narrow the type before using it
def greet_safe(uid: int) -> str:
    name = find_user(uid)
    if name is None:                          # after this check, mypy knows name is str
        return "Hello, stranger"
    return f"Hello, {name.upper()}"

print("safe, found  :", greet_safe(1))
print("safe, missing:", greet_safe(99))


# ⚠️ "Optional" is about the TYPE, not about whether the argument can be omitted
def f(a: str | None, b: str = "default") -> str:
    """`a` is required but may be None; `b` may be omitted but is never None."""
    return f"a={a!r}, b={b!r}"

print("\n" + f(None))
print(f("x", "y"))

# A mutable default, annotated correctly (see 2.7)
def add_item(item: str, basket: list[str] | None = None) -> list[str]:
    if basket is None:
        basket = []
    basket.append(item)
    return basket

print("\n", add_item("apple"), add_item("pear"))

---

## 5. `Callable` — annotating functions that take functions

You wrote decorators and `key=` functions in **4.2** and **4.4**. Annotating them needs a
type for "a function".

```
Callable[[int, str], bool]
           |    |     |
           |    |     +-- return type
           +----+-------- parameter types, in order

Callable[..., int]        any parameters, returns int
```

Import it from `collections.abc`, not `typing` — the `typing.Callable` alias is deprecated.

In [ ]:
from collections.abc import Callable
import functools

# A function that takes a function
def apply_twice(func: Callable[[int], int], value: int) -> int:
    return func(func(value))

print("apply_twice:", apply_twice(lambda n: n * 3, 2))


# A key= parameter
def sort_by(items: list[str], key: Callable[[str], int]) -> list[str]:
    return sorted(items, key=key)

print("sort_by len:", sort_by(["banana", "kiwi", "apple"], len))


# Annotating a decorator. Callable[..., X] means "any signature, returns X".
def announce(func: Callable[..., str]) -> Callable[..., str]:
    @functools.wraps(func)
    def wrapper(*args, **kwargs) -> str:
        return f">> {func(*args, **kwargs)}"
    return wrapper

@announce
def greet(name: str) -> str:
    return f"Hello, {name}"

print("\ndecorated:", greet("Aditya"))
print("name kept:", greet.__name__)


# A registry of handlers, typed
Handler = Callable[[str], str]

handlers: dict[str, Handler] = {
    "upper": str.upper,
    "lower": str.lower,
    "title": str.title,
}

for name, handler in handlers.items():
    print(f"  {name:<6} -> {handler('hello world')}")

---

## 6. `Any`, aliases, and the 3.12 `type` statement

**`Any`** switches type checking off for that value. It is compatible with everything, in
both directions. Sometimes it is the honest answer — data straight out of `json.loads()`
genuinely could be anything — but reaching for it to silence an error defeats the purpose.

**Type aliases** give a name to a complicated annotation. If you write
`dict[str, list[tuple[int, float]]]` three times, name it once.

> **Version note:** Python **3.12** added the `type` statement (PEP 695):
> `type UserId = int`. Before that, a plain assignment (`UserId = int`) or
> `UserId: TypeAlias = int` (3.10+) does the same job.

In [ ]:
import sys
from typing import Any

# Any: compatible with everything - use deliberately, not to silence errors
def parse_config(raw: dict[str, Any]) -> dict[str, Any]:
    """JSON values genuinely can be anything, so Any is honest here."""
    return {k: v for k, v in raw.items() if v is not None}

print(parse_config({"host": "localhost", "port": 8080, "debug": None}))


# Type aliases: name a shape you use repeatedly
Marks = dict[str, float]
StudentRecord = tuple[str, Marks]

def best_subject(record: StudentRecord) -> str:
    name, marks = record
    return max(marks, key=lambda k: marks[k])

record: StudentRecord = ("Aditya", {"phy": 88, "chem": 91, "maths": 79})
print("\nbest subject:", best_subject(record))


# 3.12+ `type` statement - the modern spelling
if sys.version_info >= (3, 12):
    print("\nOn 3.12+ you can write:  type Marks = dict[str, float]")
else:
    print("\nOn <3.12 use a plain assignment or TypeAlias")

print("running on:", f"{sys.version_info.major}.{sys.version_info.minor}")


# Self-documenting aliases catch mix-ups a bare `int` would not
UserId = int
OrderId = int

def fetch_order(user: UserId, order: OrderId) -> str:
    return f"order {order} for user {user}"

print("\n" + fetch_order(1, 500))
print("(a checker treats both as int here - see 17 for NewType, which does not)")

---

## 7. Actually checking: `mypy`

Annotations do nothing on their own. A **static type checker** reads them and reports
inconsistencies — without running your code.

```bash
pip install mypy
mypy your_script.py
```

Typical output:

```
script.py:12: error: Argument 1 to "add" has incompatible type "str"; expected "int"  [arg-type]
script.py:20: error: Item "None" of "str | None" has no attribute "upper"  [union-attr]
Found 2 errors in 1 file (checked 1 source file)
```

That second one is the payoff: it found a `None` bug **without executing anything**.

### Useful flags

| Flag | Effect |
|---|---|
| `mypy file.py` | Check one file |
| `--strict` | Turn on all the optional strictness |
| `--ignore-missing-imports` | Don't complain about untyped third-party libraries |
| `# type: ignore[code]` | Silence one specific line — use sparingly, and say why |

Alternatives: **pyright** (fast, powers VS Code's Pylance) and **ty**/**pyrefly** (newer).
They read the same annotations.

> Configuration lives in `pyproject.toml` — covered in **19 Tooling, Packaging and
> Environments**.

In [ ]:
# A file with a real type error, written out so you can run mypy on it yourself.
from pathlib import Path
import textwrap

sample = textwrap.dedent('''
    def find_user(uid: int) -> str | None:
        return {1: "Aditya"}.get(uid)


    def greet(uid: int) -> str:
        name = find_user(uid)
        return f"Hello, {name.upper()}"     # BUG: name may be None


    def add(a: int, b: int) -> int:
        return a + b


    print(add("2", 3))                      # BUG: str where int expected
''').lstrip()

target = Path("mypy_demo.py")
target.write_text(sample, encoding="utf-8")
print(f"wrote {target} ({len(sample.splitlines())} lines)\n")
print(sample)

print("Now run:   mypy mypy_demo.py")
print("""
Expected output:
  mypy_demo.py:7: error: Item "None" of "str | None" has no attribute "upper"  [union-attr]
  mypy_demo.py:14: error: Argument 1 to "add" has incompatible type "str"; expected "int"  [arg-type]
  Found 2 errors in 1 file (checked 1 source file)

Neither bug needs the program to run. The second one would only have shown up
in production if that line were ever reached.
""")

---

## 8. How much typing is worth it?

Type hints are **gradual** — that is the entire design. You can annotate one function, one
module, or nothing at all, and everything still runs.

A reasonable order of adoption:

| Priority | Annotate |
|---|---|
| **1. Always** | Public functions others call — your module's API |
| **2. Usually** | Anything returning `X \| None`, since that's where bugs hide |
| **3. Worth it** | Functions with non-obvious parameters (`data`, `config`, `items`) |
| **4. Optional** | Short private helpers where the types are obvious from two lines away |
| **5. Skip** | Throwaway scripts and notebook cells |

### Honest costs

- **Verbosity.** `dict[str, list[tuple[int, float]]]` is a mouthful. Use an alias.
- **A learning curve** for `Protocol`, `TypeVar` and variance (all in **17**).
- **Untyped dependencies** produce noise until you configure the checker.
- **They can lie.** Nothing stops an annotation being wrong; only running a checker keeps
  them honest. An unchecked annotation is just a comment.

> **The rule that matters:** if you annotate, run a checker in CI. Annotations that nobody
> verifies drift out of date exactly like the comments they were meant to replace.

---

## Common Mistakes & Pitfalls

1. **Believing hints are enforced.** Python ignores them at run time. Only a checker like `mypy` acts on them.
2. **Writing `typing.List[int]` in new code.** Use the builtin `list[int]` (3.9+); the `typing` aliases are deprecated.
3. **Thinking `Optional[str]` means 'this argument can be omitted'.** It means `str | None`. Optional-ness at the call site comes from having a *default*.
4. **Annotating a mutable default as `list[str] = []`.** The annotation is fine; the default is still the shared-mutable bug. Use `list[str] | None = None`.
5. **Reaching for `Any` to silence an error.** That switches checking off for that value — usually the opposite of what you wanted.
6. **Annotating `*args: tuple[int, ...]`.** Annotate the *element* type: `*args: int`.
7. **Adding annotations but never running a checker.** They then decay into comments that happen to have colons.
8. **Over-annotating internals.** Three-line private helpers rarely earn the noise.

## Best Practices

- Annotate the **public surface** first — the functions other code calls.
- Always annotate a function that can return `None`; that is where hints pay for themselves.
- Use builtin generics (`list`, `dict`, `tuple`) — not `typing.List` etc.
- Use `X | None` on 3.10+; `Optional[X]` only if you must support older versions.
- Accept `Iterable`/`Sequence` for parameters, return concrete types like `list`.
- Name repeated complex annotations with a type alias.
- Run `mypy` (or pyright) in CI so the annotations stay true.
- Adopt gradually — a partly-typed codebase is strictly better than an untyped one.

## Practice Exercises

Try these before moving on.

1. Annotate every function you wrote in **4.1**, then run `mypy` over the file.
2. Write `def first(items: Sequence[T]) -> T | None` for an empty-safe first element. (Peek at **17** for `TypeVar`.)
3. Annotate a decorator so the decorated function keeps a useful signature. What is hard about this?
4. Take `parse_config(raw: dict[str, Any])` and replace `Any` with something more precise. What do you lose and gain?
5. Write a function returning `str | None` and a caller that handles both. Introduce the bug, confirm `mypy` catches it, then fix it.
6. Create `mypy_demo.py` from the cell above, run `mypy` on it, and fix both errors.
7. Find a function in your own code whose parameter types you can't infer in 10 seconds. Annotate it — did anything surprise you?